# 📊 Marketing Campaign Response Prediction
### *Storm Satin Edition — Clean · Fixed · Production Ready*

**Pipeline:**
1. Data Loading & Cleaning
2. Feature Engineering
3. EDA (Plotly)
4. Encoding + Train/Test Split
5. Scaling
6. Random Forest Modeling
7. Evaluation Metrics
8. Business Insights
9. Feature Importance

In [ ]:
    paper_bgcolor='rgba(237,224,228,0.82)',
    plot_bgcolor='rgba(255,255,255,0.18)',
    font=dict(color='#3a1e38', family='DM Sans, sans-serif'),
    title_font=dict(color='#52357c', size=16, family='Cormorant Garamond, serif'),
    xaxis=dict(gridcolor='rgba(139,110,176,0.10)', linecolor='rgba(139,110,176,0.20)'),
    yaxis=dict(gridcolor='rgba(139,110,176,0.10)', linecolor='rgba(139,110,176,0.20)'),
    margin=dict(t=55, b=40, l=30, r=30),

## 1️⃣ Data Loading

In [ ]:
df = pd.read_csv('marketing_campaign.csv', sep='\t')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
df.info()
print('\nMissing Values:\n', df.isnull().sum()[df.isnull().sum() > 0])

## 2️⃣ Data Cleaning

In [ ]:
# Fill missing Income with median
df['Income'] = df['Income'].fillna(df['Income'].median())

# Remove duplicates
df.drop_duplicates(inplace=True)

# Strip whitespace from column names
df.columns = df.columns.str.strip()

# Drop ID column (not predictive)
df.drop('ID', axis=1, errors='ignore', inplace=True)

print(f'✅ Cleaned shape: {df.shape}')
print(f'Missing values remaining: {df.isnull().sum().sum()}')

## 3️⃣ Feature Engineering

In [ ]:
# Parse date
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], dayfirst=True)

# Age (based on 2026)
df['Age'] = 2026 - df['Year_Birth']

# Total Spending across all categories
df['Total_Spending'] = (
    df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] +
    df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds']
)

# Enrolment year & month
df['Customer_Year'] = df['Dt_Customer'].dt.year
df['Customer_Month'] = df['Dt_Customer'].dt.month

# Tenure in days
df['Customer_Tenure'] = (pd.to_datetime('today') - df['Dt_Customer']).dt.days

# Drop raw date column
df.drop('Dt_Customer', axis=1, inplace=True)

print('✅ New features: Age, Total_Spending, Customer_Year, Customer_Month, Customer_Tenure')
df[['Age', 'Total_Spending', 'Customer_Tenure']].describe()

## 4️⃣ Exploratory Data Analysis

In [ ]:
fig = px.histogram(
    df, x='Income', nbins=40,
    title='💰 Income Distribution of Customers',
    color_discrete_sequence=['#a87080']
)
fig.update_layout(**PLOT_LAYOUT)
fig.show()

In [ ]:
fig = px.pie(
    df, names='Response',
    title='🎯 Customer Response Distribution',
    color_discrete_sequence=['#6b4e96', '#c9a4ae'],
    hole=0.45
)
fig.update_layout(**PLOT_LAYOUT)
fig.show()

In [ ]:
fig = px.scatter(
    df, x='Income', y='Total_Spending',
    color='Response',
    size='Total_Spending',
    hover_data=['Age'],
    title='💎 Income vs Spending Behavior',
    color_discrete_map={0: '#c9a4ae', 1: '#6b4e96'},
    opacity=0.75
)
fig.update_layout(**PLOT_LAYOUT)
fig.show()

In [ ]:
fig = px.box(
    df, y='Age',
    title='👤 Age Distribution of Customers',
    color_discrete_sequence=['#c9a4ae']
)
fig.update_layout(**PLOT_LAYOUT)
fig.show()

In [ ]:
fig = px.violin(
    df, y='Total_Spending',
    box=True,
    title='🛒 Customer Spending Pattern',
    color_discrete_sequence=['#8b6eb0']
)
fig.update_layout(**PLOT_LAYOUT)
fig.show()

In [ ]:
fig = px.imshow(
    df.corr(numeric_only=True),
    text_auto=True,
    color_continuous_scale=[[0, '#dcc4ca'], [0.33, '#c9a4ae'], [0.66, '#a891c4'], [1, '#52357c']],
    title='🔥 Feature Correlation Heatmap'
)
fig.update_layout(**PLOT_LAYOUT)
fig.show()

## 5️⃣ Encoding

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop('Response', axis=1)
y = df_encoded['Response']

print(f'Features: {X.shape[1]}  |  Samples: {X.shape[0]}')
print(f'Class balance:\n{y.value_counts()}')

## 6️⃣ Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

## 7️⃣ Scaling

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('✅ Scaling complete')

## 8️⃣ Model Training — Random Forest

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train_sc, y_train)

y_pred = model.predict(X_test_sc)
y_prob = model.predict_proba(X_test_sc)[:, 1]

print('✅ Model trained')

## 9️⃣ Evaluation

In [ ]:
print('=' * 45)
print(f"  Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision : {precision_score(y_test, y_pred):.4f}")
print(f"  Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"  F1 Score  : {f1_score(y_test, y_pred):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test, y_prob):.4f}")
print('=' * 45)
print('\nClassification Report:\n')
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig = px.imshow(
    cm, text_auto=True,
    labels=dict(x='Predicted', y='Actual'),
    x=['No Response', 'Response'],
    y=['No Response', 'Response'],
    color_continuous_scale=[[0, '#e8e2f0'], [0.5, '#a891c4'], [1, '#52357c']],
    title='🔲 Confusion Matrix'
)
fig.update_layout(**PLOT_LAYOUT)
fig.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr, mode='lines',
    name=f'ROC (AUC = {auc_score:.3f})',
    line=dict(color='#a87080', width=2.5)
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    name='Random', line=dict(color='#c9a4ae', dash='dash', width=1.5)
))
fig.update_layout(title='📈 ROC Curve', xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate', **PLOT_LAYOUT)
fig.show()

## 🔟 Business Insights

In [ ]:
responders = df[df['Response'] == 1]
non_resp   = df[df['Response'] == 0]

print(f"{'Metric':<25} {'Responders':>14} {'Non-Responders':>16}")
print('-' * 57)
print(f"{'Avg Income':<25} {responders['Income'].mean():>14,.0f} {non_resp['Income'].mean():>16,.0f}")
print(f"{'Avg Total Spending':<25} {responders['Total_Spending'].mean():>14,.0f} {non_resp['Total_Spending'].mean():>16,.0f}")
print(f"{'Avg Age':<25} {responders['Age'].mean():>14.1f} {non_resp['Age'].mean():>16.1f}")
print(f"{'Avg Tenure (days)':<25} {responders['Customer_Tenure'].mean():>14.0f} {non_resp['Customer_Tenure'].mean():>16.0f}")

## 1️⃣1️⃣ Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
top15 = importances.sort_values(ascending=True).tail(15)

fig = px.bar(
    top15, orientation='h',
    color=top15.values,
    color_continuous_scale=[[0, '#dcc4ca'], [0.33, '#c9a4ae'], [0.66, '#a891c4'], [1, '#52357c']],
    title='🌳 Top 15 Feature Importances'
)
fig.update_layout(**PLOT_LAYOUT)
fig.show()